In [ ]:
# this file has been used to test a lot of different ideas and none of them have been removed

#test with the RDD

from pyspark.sql import SparkSession

#spark = SparkSession.builder \
 #   .appName("ReadDataFromS3") \
  #  .config("spark.hadoop.fs.s3a.aws.credentials.provider", "com.amazonaws.auth.DefaultAWSCredentialsProviderChain") \
   # .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    #.config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.1") \
    #.getOrCreate()

spark = SparkSession.builder \
    .appName("Fractal") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider") \
    .config("spark.hadoop.fs.s3a.access.key", "") \
    .config("spark.hadoop.fs.s3a.secret.key", "") \
    .config("spark.hadoop.fs.s3a.connection.timeout", "50000") \
    .config("spark.hadoop.fs.s3a.threads.keepalivetime", "60000") \
    .config("spark.hadoop.fs.s3a.connection.establish.timeout", "30000") \
    .config("spark.hadoop.fs.s3a.multipart.purge.age", "30000000") \
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.1") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/01/08 11:17:22 WARN Utils: Your hostname, Omowonuolas-MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 10.2.190.9 instead (on interface en0)
26/01/08 11:17:22 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/Users/omowonuolaakintola/anaconda3/envs/bigdata/lib/python3.14/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /Users/omowonuolaakintola/.ivy2.5.2/cache
The jars for the packages stored in: /Users/omowonuolaakintola/.ivy2.5.2/jars
org.apache.hadoop#hadoop-aws added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-91111bfe-5f4f-4507-8ba1-4f7a7ac79377;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.3.1 in central
	found com.amazonaws#aws-java-sdk-bundle;1.11.901 in central
	found org.wildfly.openssl#wildfly-openssl;

In [2]:
!pip install boto3 # https://arrow.apache.org/docs/python/install.html

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.6/14.6 MB 3.3 MB/s  0:00:04 eta 0:00:01
  Attempting uninstall: botocore
    Found existing installation: botocore 1.41.5
    Uninstalling botocore-1.41.5:
      Successfully uninstalled botocore-1.41.5
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [boto3]re]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
aiobotocore 2.26.0 requires botocore<1.41.6,>=1.41.0, but you have botocore 1.42.24 which is incompatible.


In [3]:
import boto3
import pyarrow.parquet as pq
import pandas as pd
from io import BytesIO

# Setup
s3 = boto3.client('s3')

# 1. Read from S3
print("Reading metadata from S3...")
response = s3.get_object(Bucket='ubs-datasets', Key='bigearthnet/metadata.parquet')
metadata_df = pq.read_table(BytesIO(response['Body'].read())).to_pandas()
print(f"Loaded {len(metadata_df)} rows")

# 2. Add path columns
print("Adding S3 paths...")
metadata_df["s1_path"] = (
    "s3://ubs-datasets/bigearthnet/BigEarthNet-S1/"
    + metadata_df["s1_name"].str.extract(r'^(S1[AB]_IW_GRDH_1SDV_\d{8}T\d{6})')[0]
    + "/" + metadata_df["s1_name"] + "/"
)
metadata_df["s2_path"] = (
    "s3://ubs-datasets/bigearthnet/BigEarthNet-S2/"
    + metadata_df["patch_id"].str.extract(r'^(S2[AB]_MSIL2A_\d{8}T\d{6}_N\d{4}_R\d{3}_T\d{2}[A-Z]{3})')[0]
    + "/" + metadata_df["patch_id"] + "/"
)
metadata_df["ref_path"] = (
    "s3://ubs-datasets/bigearthnet/Reference_Maps/"
    + metadata_df["patch_id"].str.extract(r'^(S2[AB]_MSIL2A_\d{8}T\d{6}_N\d{4}_R\d{3}_T\d{2}[A-Z]{3})')[0]
    + "/" + metadata_df["patch_id"] + "/"
)

# 3. Validate a sample
print("\nSample paths:")
print(f"  S1: {metadata_df['s1_path'].iloc[0]}")
print(f"  S2: {metadata_df['s2_path'].iloc[0]}")
print(f"  Ref: {metadata_df['ref_path'].iloc[0]}")

# 4. Write back to S3
#print("\nUploading to S3...")
#buffer = BytesIO()
#metadata_df.to_parquet(buffer, engine='pyarrow', index=False)
#buffer.seek(0)

#s3.put_object(
  #  Bucket='ubs-homes',
  #  Key='erasmus/ola/metadata/metadata_updated.parquet',
  #  Body=buffer.getvalue()
#)
#print("Done!")

Reading metadata from S3...
Loaded 480038 rows
Adding S3 paths...

Sample paths:
  S1: s3://ubs-datasets/bigearthnet/BigEarthNet-S1/S1B_IW_GRDH_1SDV_20170612T165809/S1B_IW_GRDH_1SDV_20170612T165809_33UUP_26_57/
  S2: s3://ubs-datasets/bigearthnet/BigEarthNet-S2/S2A_MSIL2A_20170613T101031_N9999_R022_T33UUP/S2A_MSIL2A_20170613T101031_N9999_R022_T33UUP_26_57/
  Ref: s3://ubs-datasets/bigearthnet/Reference_Maps/S2A_MSIL2A_20170613T101031_N9999_R022_T33UUP/S2A_MSIL2A_20170613T101031_N9999_R022_T33UUP_26_57/


In [13]:

#df_train = spark.read.parquet("s3a://ubs-datasets/FRACTAL/data/train/")
#df_val = spark.read.parquet("s3a://ubs-datasets/FRACTAL/data/val")
#df_test = spark.read.parquet("s3a://ubs-datasets/FRACTAL/data/test")

data = spark.read.parquet("s3a://ubs-datasets/bigearthnet/metadata.parquet")

# Explore the dataset
print('Data overview')
data.show(truncate=False)

#print('Columns overview')
#df_train.printSchema()


#s1/folder/FOLDER

26/01/08 11:17:29 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


Data overview


+--------------------------------------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----+-------+--------------------------------------------+--------------------------------+----------------------+------------------------+
|patch_id                                          |labels                                                                                                                                                                                   |split|country|s1_name                                     |s2v1_name                       |contains_seasonal_snow|contains_cloud_or_shadow|
+--------------------------------------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----

In [1]:
# Installing pyarrow package
!pip install pyarrow # https://arrow.apache.org/docs/python/install.html

In [18]:
import pyarrow.parquet as pq

In [4]:
# Reading and writing parquet file source: https://arrow.apache.org/docs/python/parquet.html
# Read the metadata Parquet file using PyArrow and convert it to a pandas DataFrame
metadata_df = pq.read_table('./metadata.parquet').to_pandas()

# Preview the first 5 rows of the metadata
metadata_df.head(5)

,patch_id,labels,split,country,s1_name,s2v1_name,contains_seasonal_snow,contains_cloud_or_shadow
0,S2A_MSIL2A_20170613T101031_N9999_R022_T33UUP_2...,"[Arable land, Broad-leaved forest, Mixed fores...",test,Austria,S1B_IW_GRDH_1SDV_20170612T165809_33UUP_26_57,S2A_MSIL2A_20170613T101031_26_57,False,False
1,S2A_MSIL2A_20170613T101031_N9999_R022_T33UUP_2...,"[Arable land, Broad-leaved forest, Inland wate...",test,Austria,S1B_IW_GRDH_1SDV_20170612T165809_33UUP_27_55,S2A_MSIL2A_20170613T101031_27_55,False,False
2,S2A_MSIL2A_20170613T101031_N9999_R022_T33UUP_2...,"[Arable land, Broad-leaved forest, Coniferous ...",test,Austria,S1B_IW_GRDH_1SDV_20170612T165809_33UUP_27_56,S2A_MSIL2A_20170613T101031_27_56,False,False
3,S2A_MSIL2A_20170613T101031_N9999_R022_T33UUP_2...,"[Broad-leaved forest, Complex cultivation patt...",test,Austria,S1B_IW_GRDH_1SDV_20170612T165809_33UUP_27_57,S2A_MSIL2A_20170613T101031_27_57,False,False
4,S2A_MSIL2A_20170613T101031_N9999_R022_T33UUP_2...,"[Broad-leaved forest, Complex cultivation patt...",test,Austria,S1B_IW_GRDH_1SDV_20170612T165809_33UUP_27_58,S2A_MSIL2A_20170613T101031_27_58,False,False


In [5]:
# Construct S3 paths for Sentinel-2 data
metadata_df["s1_path"]  = (
    "s3://ubs-datasets/bigearthnet/BigEarthNet-S1/"
    + metadata_df["s1_name"].str.extract(
        r'^(S1[AB]_IW_GRDH_1SDV_\d{8}T\d{6})' #S1B_IW_GRDH_1SDV_20170612T165809_33UUP_26_57
      )[0]
    + "/"
    + metadata_df["s1_name"]
    + "/"
)
# Construct S3 paths for Sentinel-2 data
metadata_df["s2_path"]  = (
    "s3://ubs-datasets/bigearthnet/BigEarthNet-S2/"
    + metadata_df["patch_id"].str.extract(
        r'^(S2[AB]_MSIL2A_\d{8}T\d{6}_N\d{4}_R\d{3}_T\d{2}[A-Z]{3})' # S2A_MSIL2A_20170613T101031_N9999_R022_T33UUP_26_57
      )[0]
    + "/"
    + metadata_df["patch_id"]
    + "/"
)
# Construct S3 paths for reference maps.
metadata_df["ref_path"]  = (
    "s3://ubs-datasets/bigearthnet/Reference_Maps/"
    + metadata_df["patch_id"].str.extract(
        r'^(S2[AB]_MSIL2A_\d{8}T\d{6}_N\d{4}_R\d{3}_T\d{2}[A-Z]{3})' # S2A_MSIL2A_20170613T101031_N9999_R022_T33UUP_26_57
      )[0]
    + "/"
    + metadata_df["patch_id"]
    + "/"
)

In [6]:
# printing first rows path to test
print(metadata_df["s1_path"][0])
print(metadata_df["s2_path"][0])
print(metadata_df["ref_path"][0])

s3://ubs-datasets/bigearthnet/BigEarthNet-S1/S1B_IW_GRDH_1SDV_20170612T165809/S1B_IW_GRDH_1SDV_20170612T165809_33UUP_26_57/
s3://ubs-datasets/bigearthnet/BigEarthNet-S2/S2A_MSIL2A_20170613T101031_N9999_R022_T33UUP/S2A_MSIL2A_20170613T101031_N9999_R022_T33UUP_26_57/
s3://ubs-datasets/bigearthnet/Reference_Maps/S2A_MSIL2A_20170613T101031_N9999_R022_T33UUP/S2A_MSIL2A_20170613T101031_N9999_R022_T33UUP_26_57/


In [7]:
# Exporting updated metada into parquet file 
metadata_df.to_parquet(
    "./metadata_updated.parquet",  # output file path
    engine="pyarrow",                        # can also use "fastparquet"
    index=False                               # don’t save the DataFrame index
)

In [5]:
# Quick check of updated parquet file
df = pq.read_table("./metadata_updated.parquet").to_pandas().head(5)

In [6]:
print(df.columns.tolist())

['patch_id', 'labels', 'split', 'country', 's1_name', 's2v1_name', 'contains_seasonal_snow', 'contains_cloud_or_shadow', 's1_path', 's2_path', 'ref_path']
